# 02c_check_cellscape_xenium_match: do the CellScape cores really match the Xenium cores?

```text
Independent check of 02b. The CellScape and Xenium sections are serial sections of the
same block, so the tissue inside every core (its outline, tears, holes, fragments) looks
the same on both -- but only with the right orientation and the right core label. For every
core found on both sections this script compares the CellScape DNA image of the core with
the Xenium cells of the core:

  1. orientation test: the CellScape core in all 8 orientations (0/90/180/270 degrees, with
     and without mirroring; each with a small extra tilt and shift search) against the
     Xenium core -> which orientation fits best, core by core. If 02b is right, nearly all
     cores pick the orientation 02b chose.
  2. label test: the Xenium core against its own CellScape core and against the 8 CellScape
     cores around it (same orientation) -> the own core should fit best. A grid that is one
     row or column off fails this test.

CellScape image: either the <slide>_TMA_preview.png pathology sent (the grid drawn over the
DNA channel; the coloured circles and labels are ignored), or, better, a DNA image of the
CellScape scan as OME-TIFF (any pyramid level is read; give --cellscape-channel if it has
several channels). For the PNG the full CellScape image size is needed to convert pixels to
um: known for FL0026 / FL0027 (pathology README), otherwise --cellscape-size-px WxH.

Outputs (next to the 02b grid, default <scripts>/config/core_grid/)
  <slide_id>_match_summary.png   orientation votes, own core vs neighbours, one core in all 8
                                 orientations
  <slide_id>_match_gallery.png   CellScape (magenta) over Xenium (green), white = agreement
  <slide_id>_match_per_core.csv  per core: score per orientation, best orientation, own vs
                                 best neighbour score

Example
  python 02c_check_cellscape_xenium_match.py --slide-id TMA_ORGAN --sector A \
      --cores-csv FL0027_TMA_cores.csv --cellscape-image FL0027_TMA_preview.png \
      --xenium-dir /path/to/output-XETG...
```

**Notebook version of `experiment1_tma/02c_check_cellscape_xenium_match.py`.** Same code, laid out for interactive work:

1. Set the options in the **Parameters** cell (the script's command-line options: `--slide-id` becomes `slide_id`).
2. Run the cells in order. After each cell you can inspect its variables, tables and images.

Use the `.py` script for batch jobs on the cluster; both produce the same outputs. This notebook is generated by `tools/make_notebooks.py`, which overwrites it, so save your own edits under another name.

In [ ]:
import os

# Folder with 02c_check_cellscape_xenium_match.py. Normally this notebook sits in
# notebooks/ next to the scripts ('..'); it is also found if the notebook is in the script
# folder itself or deeper, or if the scripts are up to 3 subfolders below the notebook.
# If not found, set SCRIPT_DIR yourself, e.g.
# SCRIPT_DIR = '/work_ikmb/<user>/Spatial_TMA/Scripts_tma'
SCRIPT_DIR = None
# a cell run while SCRIPT_DIR was still None leaves None on the import path, which breaks
# later imports (scipy: 'stat: path should be string ... not NoneType'): drop it
import sys
sys.path[:] = [p for p in sys.path if isinstance(p, str)]
_required = ['02c_check_cellscape_xenium_match.py']
_cands = [os.path.abspath('..')] + [os.path.abspath(os.path.join(*['..'] * k) if k else '.') for k in range(4)]
if SCRIPT_DIR is None:
    SCRIPT_DIR = next((d for d in _cands if all(os.path.exists(os.path.join(d, f)) for f in _required)), None)
if SCRIPT_DIR is None:
    # the scripts may also be in a subfolder (e.g. an unzipped experiment1_tma/): newest copy wins
    import glob
    _below = sorted({os.path.dirname(os.path.abspath(p)) for k in range(1, 4)
                     for p in glob.glob(os.path.join(*['*'] * k, _required[0]))
                     if '.ipynb_checkpoints' not in p},
                    key=lambda d: os.path.getmtime(os.path.join(d, _required[0])), reverse=True)
    _ok = [d for d in _below if all(os.path.exists(os.path.join(d, f)) for f in _required)]
    _cands += _below
    if _ok:
        SCRIPT_DIR = _ok[0]
        if len(_ok) > 1:
            print('Several copies of the scripts below this folder; using the newest:', *_ok, sep='\n  ')
if SCRIPT_DIR is None or not all(os.path.exists(os.path.join(SCRIPT_DIR, f)) for f in _required):
    _near = next((d for d in _cands if os.path.exists(os.path.join(d, _required[0]))), None)
    raise FileNotFoundError(
        (f'Found {_required[0]} in {_near}, but not {_required[1:]} there -- copy the py/ folder of the '
         'repository next to the scripts.' if _near else
         f'Could not find {_required[0]} in {sorted(set(_cands))}. Put this notebook in the notebooks/ '
         'folder next to the scripts, or set SCRIPT_DIR at the top of this cell.')
        + f'\n(this notebook runs in {os.getcwd()})')
print('Scripts folder:', SCRIPT_DIR)

# Which Python runs this notebook? It must be the conda environment, not the system one.
import sys
print('Python:', sys.executable)
try:
    import numpy  # noqa: F401
except ModuleNotFoundError:
    # find the environment's own python, so the fix below names the exact path
    cands = [os.path.expanduser('~/.conda/envs/exp1-imaging/bin/python'),
             os.path.join(sys.prefix, 'envs', 'exp1-imaging', 'bin', 'python')]
    cands += [os.path.join(d, 'exp1-imaging', 'bin', 'python')
              for d in os.environ.get('CONDA_ENVS_PATH', '').split(os.pathsep) if d]
    found = [c for c in cands if os.path.exists(c)]
    py = found[0] if found else '"$CONDA_PREFIX/bin/python"'
    raise ModuleNotFoundError(
        'This kernel runs ' + sys.executable + ', not the exp1-imaging environment.\n'
        'If you already picked "exp1-imaging" under Kernel > Change kernel, that kernel was registered\n'
        'with the wrong Python. Fix it once in a terminal (re-registers it with the right one):\n'
        '  conda activate exp1-imaging\n'
        f'  {py} -m pip install ipykernel\n'
        f'  {py} -m ipykernel install --user --name exp1-imaging --display-name exp1-imaging\n'
        'then reload the Jupyter page, Kernel > Change kernel > exp1-imaging, run from the top.\n'
        '(environment file: experiment1_tma/env/py_imaging.yml)')

# Packages this step needs (import name: pip name). Anything missing is installed INTO THIS
# kernel with %pip in a notebook cell -- not with pip in a terminal, which may be another Python.
import importlib.util
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib', 'PIL': 'pillow', 'skimage': 'scikit-image', 'tifffile': 'tifffile'}
missing = [pip for mod, pip in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError(
        f'Missing in this kernel: {", ".join(missing)}\n'
        f'Run this in a new cell:  %pip install {" ".join(missing)}\n'
        'then Kernel > Restart, and run the notebook again from the top.')
print('All packages found.')

In [ ]:
import argparse
import json
import math
import os
import re

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.spatial import cKDTree

HERE = SCRIPT_DIR
CS_UM_PER_PX = 0.18                                     # CellScape full-resolution pixel size
KNOWN_SIZES = {"FL0026": (174689, 91289), "FL0027": (179811, 91151)}    # pathology README

## Parameters

In [ ]:
from types import SimpleNamespace

# Same options as `python experiment1_tma/02c_check_cellscape_xenium_match.py --help` (--slide-id -> slide_id).

# Options marked REQUIRED have no default: replace their None, e.g. slide_id="TMA_ORGAN".
# Paths are plain strings in quotes.

a = SimpleNamespace(
    slide_id=None,           # REQUIRED. Xenium slide_id, as for 02b
    cores_csv=None,          # REQUIRED. <slide>_TMA_cores.csv of the CellScape dearray
    sector=None,             # REQUIRED. sector letter(s), as for 02b, e.g. ['A']. a list
    cellscape_image=None,    # REQUIRED. <slide>_TMA_preview.png, or a CellScape DNA image (OME-TIFF)
    xenium_dir=None,         # REQUIRED. Xenium output folder (output-XETG...) of this slide
    grid_dir=None,           # folder with the 02b outputs (<slide_id>.csv + _cellscape_to_xenium.json); default <scripts>/config/core_grid
    cellscape_size_px=None,  # full CellScape image size 'WxH' (only for a PNG preview; known for FL0026/FL0027)
    cellscape_channel=0,     # channel of the OME-TIFF with DNA
    resolution_um=10.0,      # pixel size of the comparison
    max_shift_um=200.0,      # shift search per core
    n_gallery=48,            # cores shown in the gallery
)

missing = [k for k in ('slide_id', 'cores_csv', 'sector', 'cellscape_image', 'xenium_dir') if getattr(a, k) in (None, '')]
if missing:
    raise ValueError(f'Fill in these options in the cell above (replace None with a value): {missing}')

In [ ]:
# Options not set in the Parameters cell (e.g. a cell copied from an older version
# of this notebook) get the script's default. Nothing to edit here.
_defaults = dict(
    slide_id=None,
    cores_csv=None,
    sector=None,
    cellscape_image=None,
    xenium_dir=None,
    grid_dir=None,
    cellscape_size_px=None,
    cellscape_channel=0,
    resolution_um=10.0,
    max_shift_um=200.0,
    n_gallery=48,
)
for _k, _v in _defaults.items():
    if not hasattr(a, _k):
        setattr(a, _k, _v)
        print(f'{_k} not in the Parameters cell -> default {_v!r}')
_missing = [k for k in ('slide_id', 'cores_csv', 'sector', 'cellscape_image', 'xenium_dir') if getattr(a, k, None) in (None, '')]
if _missing:
    raise ValueError(f'Still empty: {_missing}. Fill them in the Parameters cell, run THAT cell '
                     '(Shift+Enter), then this one.')
print('Options in use:', {k: v for k, v in vars(a).items() if v not in (None, False)})

## Helper functions

### Images

In [ ]:
def fill_masked(img, valid, sigma):
    """Replace invalid pixels (drawn circles / text) by the blurred valid neighbourhood."""
    num = ndimage.gaussian_filter(np.where(valid, img, 0.0), sigma)
    den = ndimage.gaussian_filter(valid.astype(float), sigma)
    out = img.copy()
    out[~valid] = (num / np.maximum(den, 1e-6))[~valid]
    return out

In [ ]:
def ring_score(overlay, cores, um_x, um_y, ox, oy):
    """Fraction of points on the expected core circles that fall on drawn (coloured) pixels."""
    th = np.linspace(0, 2 * np.pi, 48, endpoint=False)
    r = cores["diameter_um"].to_numpy()[:, None] / 2
    x = (cores["centroid_x_um"].to_numpy()[:, None] + r * np.cos(th)) / um_x + ox
    y = (cores["centroid_y_um"].to_numpy()[:, None] + r * np.sin(th)) / um_y + oy
    return float(ndimage.map_coordinates(overlay, [y.ravel(), x.ravel()], order=1, mode="constant").mean())

In [ ]:
def load_preview(path, size_px, all_cores):
    from PIL import Image
    rgb = np.asarray(Image.open(path).convert("RGB")).astype(np.float64)
    nonwhite = rgb.min(-1) < 235
    rows = np.where(nonwhite.mean(1) > 0.9)[0]
    cols = np.where(nonwhite.mean(0) > 0.9)[0]
    if len(rows) < 10 or len(cols) < 10:
        raise ValueError(f"{path}: could not find the image area inside the figure")
    r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1
    rgb = rgb[r0:r1, c0:c1]
    W, H = size_px
    um_x = W * CS_UM_PER_PX / rgb.shape[1]
    um_y = H * CS_UM_PER_PX / rgb.shape[0]
    if abs(um_x / um_y - 1) > 0.03:
        print(f"WARNING: preview aspect does not match the CellScape size {W}x{H} ({um_x:.2f} vs {um_y:.2f} um/px)")
    sat = rgb.max(-1) - rgb.min(-1)
    overlay = sat > 40                                    # yellow / red circles and labels
    gray = rgb.mean(-1)
    # check (and fine-tune) the pixel <-> um calibration on the drawn circles
    ov = ndimage.gaussian_filter(overlay.astype(float), 1.0)
    best = (ring_score(ov, all_cores, um_x, um_y, 0, 0), 1.0, 0.0, 0.0)
    for f in np.linspace(0.985, 1.015, 13):
        for ox in range(-3, 4):
            for oy in range(-3, 4):
                sc = ring_score(ov, all_cores, um_x * f, um_y * f, ox, oy)
                if sc > best[0]:
                    best = (sc, f, ox, oy)
    sc, f, ox, oy = best
    base = float(overlay.mean())
    print(f"CellScape preview: {rgb.shape[1]}x{rgb.shape[0]} px, {um_x * f:.1f} um/px; the drawn core circles "
          f"are found at the expected places ({100 * sc:.0f}% of circle points on drawn pixels, background "
          f"{100 * base:.0f}%)")
    if sc < 3 * base:
        print("WARNING: the drawn circles do not line up with the core table -- is this the preview of the same "
              "slide, and in file orientation?")
    return gray, ~overlay, (um_x * f, um_y * f), (-ox * um_x * f, -oy * um_y * f)

In [ ]:
def load_ome(path, channel, target_um):
    import tifffile
    with tifffile.TiffFile(path) as tf:
        s = tf.series[0]
        levels = s.levels if hasattr(s, "levels") else [s]
        axes = s.axes
        full_w = s.shape[axes.index("X")]
        k = 0
        for i, lv in enumerate(levels):
            if full_w / lv.shape[axes.index("X")] * CS_UM_PER_PX <= target_um:
                k = i
        arr = levels[k].asarray()
        w = levels[k].shape[axes.index("X")]
    if "C" in axes:
        arr = np.take(arr, channel, axis=axes.index("C"))
    arr = np.squeeze(arr).astype(np.float64)
    um = full_w / w * CS_UM_PER_PX
    print(f"CellScape image: level {k}, {arr.shape[1]}x{arr.shape[0]} px, {um:.2f} um/px")
    return arr, np.ones(arr.shape, bool), (um, um), (0.0, 0.0)

In [ ]:
def xenium_density(xy, b):
    x0, y0 = xy.min(0) - 50 * b
    nx = int(math.ceil((xy[:, 0].max() - x0) / b)) + 50
    ny = int(math.ceil((xy[:, 1].max() - y0) / b)) + 50
    h, _, _ = np.histogram2d(xy[:, 1], xy[:, 0], bins=[ny, nx], range=[[y0, y0 + ny * b], [x0, x0 + nx * b]])
    return ndimage.gaussian_filter(h, 1.0), (x0, y0)

### Comparison

In [ ]:
def orientations():
    out = []
    for mirror in (False, True):
        for k in range(4):
            a = math.radians(90 * k)
            r = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]).round(12)
            out.append((f"rot{90 * k}{'+mirror' if mirror else ''}", r @ np.diag([-1.0, 1.0]) if mirror else r))
    return out

In [ ]:
def rot(deg):
    a = math.radians(deg)
    return np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])

In [ ]:
def sample(img, origin, um, pts):
    """Image value at points given in um (bilinear)."""
    cols = (pts[..., 0] - origin[0]) / um[0] - 0.5
    rows = (pts[..., 1] - origin[1]) / um[1] - 0.5
    return ndimage.map_coordinates(img, [rows, cols], order=1, mode="constant", cval=0.0)

In [ ]:
def norm_patch(p):
    p = np.log1p(np.maximum(p, 0))
    sd = p.std()
    return (p - p.mean()) / sd if sd > 0 else p * 0

In [ ]:
def best_ncc(big, small):
    """Max normalised cross-correlation of `small` over all positions inside `big`."""
    from skimage.feature import match_template
    if small.std() == 0 or big.std() == 0:
        return -1.0, (0, 0)
    m = match_template(big, small)
    i = np.unravel_index(int(np.argmax(m)), m.shape)
    return float(m[i]), i

In [ ]:
def patches(cs, cs_origin, cs_um, xen, xen_origin, b, Pc, Xc, L, half, pad):
    """Xenium patch around Xc, and a function giving the CellScape patch (with search margin)
    around Pc for a given CellScape->Xenium linear map."""
    u = np.arange(-half, half + 1e-9, b)
    gx, gy = np.meshgrid(u, u)
    grid = np.stack([gx, gy], -1)
    xp = norm_patch(sample(xen, xen_origin, (b, b), Xc + grid))
    ub = np.arange(-half - pad, half + pad + 1e-9, b)
    bx, by = np.meshgrid(ub, ub)
    big = np.stack([bx, by], -1)

    def cs_patch(M):
        Minv = np.linalg.inv(M)
        pts = Pc + big @ Minv.T                         # Xenium-frame offsets -> CellScape um
        return norm_patch(sample(cs, cs_origin, cs_um, pts))
    return xp, cs_patch

In [ ]:
def overlay_tile(xp, cp):
    def unit(p):
        lo, hi = np.percentile(p, [5, 99.5])
        return np.clip((p - lo) / max(hi - lo, 1e-9), 0, 1)
    c, x = unit(cp), unit(xp)
    return np.dstack([c, x, c])                         # CellScape magenta, Xenium green, both white

In [ ]:
def plot_summary(slide_id, res, votes, chosen, names, tiles, cs, cs_origin, cs_um, xen, xen_origin, b, both,
                 cs_cores, L, half, out_dir):
    import matplotlib
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 8, height_ratios=[1.1, 1])
    ax = fig.add_subplot(gs[0, :4])
    cols = ["#2a78d6" if k == chosen else "#b8b8b8" for k in names]
    ax.bar(range(8), votes.to_numpy(), color=cols)
    ax.set_xticks(range(8))
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("cores fitting best")
    ax.set_title(f"Orientation test: which orientation makes each core's tissue match? (n = {len(res)})\n"
                 f"blue = the orientation 02b chose ({chosen})", fontsize=10)
    for i, v in enumerate(votes.to_numpy()):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
    ax2 = fig.add_subplot(gs[0, 4:])
    ax2.scatter(res["best_neighbour_score"], res["own_score"], s=12, c="#2a78d6", alpha=0.7)
    lim = [min(res[["own_score", "best_neighbour_score"]].min().min(), 0) - 0.05, 1.0]
    ax2.plot(lim, lim, color="#888", lw=1, ls="--")
    ax2.set_xlim(lim)
    ax2.set_ylim(lim)
    ax2.set_xlabel("best of the 8 neighbouring CellScape cores")
    ax2.set_ylabel("own CellScape core (same name)")
    wins = int((res["own_score"] > res["best_neighbour_score"]).sum())
    ax2.set_title(f"Label test: shape match with own core vs neighbours\nabove the line = own core fits best "
                  f"({wins} of {len(res)})", fontsize=10)
    # one example core in all 8 orientations
    ex = res.loc[(res[[f"score_{k}" for k in names]].max(1) - res[[f"score_{k}" for k in names]].median(1)).idxmax()]
    r = both[both["core_name"] == ex["core_name"]].iloc[0]
    Pc = cs_cores.loc[r["cellscape_core"], ["centroid_x_um", "centroid_y_um"]].to_numpy(dtype=float)
    Xc = np.array([r["x_um"], r["y_um"]])
    xp, cs_patch = patches(cs, cs_origin, cs_um, xen, xen_origin, b, Pc, Xc, L, half, 200.0)
    for i, (name, M) in enumerate(orientations()):
        big = cs_patch(L @ M)
        sc, (p, q) = best_ncc(big, xp)
        axi = fig.add_subplot(gs[1, i])
        axi.imshow(overlay_tile(xp, big[p:p + xp.shape[0], q:q + xp.shape[1]]))
        axi.set_title(f"{name}\nscore {sc:.2f}", fontsize=8, color="#2a78d6" if name == chosen else "black")
        axi.axis("off")
    fig.text(0.5, 0.02, f"core {ex['core_name']} ({ex['cellscape_core']}): CellScape DNA (magenta) over Xenium "
             f"cells (green) in each orientation -- white = tissue on both", ha="center", fontsize=9)
    fig.suptitle(f"{slide_id}: does the CellScape grid match the Xenium cores?", fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.96])
    fig.savefig(os.path.join(out_dir, f"{slide_id}_match_summary.png"), dpi=150)

In [ ]:
def plot_gallery(slide_id, tiles, chosen, n_show, out_dir):
    import matplotlib
    import matplotlib.pyplot as plt
    if not tiles:
        return
    idx = np.linspace(0, len(tiles) - 1, min(n_show, len(tiles))).round().astype(int)
    sel = [tiles[i] for i in idx]
    ncol = 8
    nrow = int(math.ceil(len(sel) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(2 * ncol, 2.15 * nrow))
    for ax in np.atleast_1d(axs).ravel():
        ax.axis("off")
    for ax, (name, sc, tile) in zip(np.atleast_1d(axs).ravel(), sel):
        ax.imshow(tile)
        ax.set_title(f"{name}  {sc:.2f}", fontsize=8)
    fig.suptitle(f"{slide_id}: CellScape DNA (magenta) over Xenium cells (green), orientation {chosen}; "
                 f"white = tissue on both. Score = shape match (1 = identical)", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(out_dir, f"{slide_id}_match_gallery.png"), dpi=150)

## Run

In [ ]:
if isinstance(a.sector, str):
    a.sector = [t for t in re.split(r"[\s,;]+", a.sector.strip()) if t]
grid_dir = a.grid_dir or os.path.join(HERE, "config", "core_grid")
grid = pd.read_csv(os.path.join(grid_dir, f"{a.slide_id}.csv"))
with open(os.path.join(grid_dir, f"{a.slide_id}_cellscape_to_xenium.json")) as fh:
    info = json.load(fh)
A = np.array(info["A"])
M_by_name = dict(orientations())
chosen = info["orientation"]
L = A @ np.linalg.inv(M_by_name[chosen])            # scale + tilt part of the 02b transform

In [ ]:
all_cores = pd.read_csv(a.cores_csv)
cs_cores = all_cores[all_cores["sector"].astype(str).isin(a.sector)].set_index("core")

# images
if a.cellscape_image.lower().endswith((".png", ".jpg", ".jpeg")):
    size = None
    if a.cellscape_size_px:
        size = tuple(int(v) for v in re.split(r"[x, ]+", a.cellscape_size_px.strip()))
    else:
        for key, v in KNOWN_SIZES.items():
            if key in os.path.basename(a.cellscape_image) or key in os.path.basename(a.cores_csv):
                size = v
    if size is None:
        raise ValueError("give --cellscape-size-px WxH (full CellScape image size in pixels)")
    cs, valid, cs_um, cs_origin = load_preview(a.cellscape_image, size, all_cores)
else:
    cs, valid, cs_um, cs_origin = load_ome(a.cellscape_image, a.cellscape_channel, a.resolution_um)
cs = fill_masked(cs, valid, 2.0)
cs = ndimage.gaussian_filter(cs, max(0.5, a.resolution_um / max(cs_um) / 2))

In [ ]:
xyf = os.path.join(a.xenium_dir, "cells.parquet")
xy = pd.read_parquet(xyf, columns=["x_centroid", "y_centroid"]).to_numpy(np.float64) if os.path.isfile(xyf) \
    else pd.read_csv(os.path.join(a.xenium_dir, "cells.csv.gz"), usecols=["x_centroid", "y_centroid"]).to_numpy()
b = max(a.resolution_um, min(cs_um))
xen, xen_origin = xenium_density(xy, b)

In [ ]:
both = grid[(grid["presence"] == "both") & grid["cellscape_core"].isin(cs_cores.index)].reset_index(drop=True)
P_all = cs_cores[["centroid_x_um", "centroid_y_um"]].to_numpy()
nb_tree = cKDTree(P_all)
spacing = float(np.median(nb_tree.query(P_all, k=2)[0][:, 1]))
half = 0.55 * spacing * math.sqrt(abs(np.linalg.det(L)))
pad = a.max_shift_um
tilts = (-4.0, 0.0, 4.0)
names = [n for n, _ in orientations()]
print(f"{len(both)} cores with tissue on both sections are compared "
      f"(patch {2 * half:.0f} um, {b:.0f} um pixels, shift search +-{pad:.0f} um)")

In [ ]:
rows, tiles = [], []
for r in both.itertuples(index=False):
    Pc = cs_cores.loc[r.cellscape_core, ["centroid_x_um", "centroid_y_um"]].to_numpy(dtype=float)
    Xc = np.array([r.x_um, r.y_um])
    xp, cs_patch = patches(cs, cs_origin, cs_um, xen, xen_origin, b, Pc, Xc, L, half, pad)
    score = {}
    for name, M in orientations():
        score[name] = max(best_ncc(cs_patch(rot(t) @ L @ M), xp)[0] for t in tilts)
    # label test: own core vs the 8 CellScape cores around it, in the chosen orientation
    M = M_by_name[chosen]
    _, nbr = nb_tree.query(Pc, k=9)
    own = score[chosen]
    others = []
    for j in nbr:
        name_j = cs_cores.index[j]
        if name_j == r.cellscape_core:
            continue
        _, cs_patch_j = patches(cs, cs_origin, cs_um, xen, xen_origin, b, P_all[j], Xc, L, half, pad)
        others.append(max(best_ncc(cs_patch_j(rot(t) @ L @ M), xp)[0] for t in tilts))
    best_name = max(score, key=score.get)
    rows.append(dict(core_name=r.core_name, cellscape_core=r.cellscape_core,
                     best_orientation=best_name, **{f"score_{k}": round(v, 3) for k, v in score.items()},
                     own_score=round(own, 3), best_neighbour_score=round(max(others), 3)))
    # tile for the gallery (chosen orientation, best tilt / shift)
    big = cs_patch(L @ M)
    _, (i, j) = best_ncc(big, xp)
    tiles.append((r.core_name, own, overlay_tile(xp, big[i:i + xp.shape[0], j:j + xp.shape[1]])))

In [ ]:
res = pd.DataFrame(rows)
res.to_csv(os.path.join(grid_dir, f"{a.slide_id}_match_per_core.csv"), index=False)

votes = res["best_orientation"].value_counts().reindex(names, fill_value=0)
n = len(res)
own_wins = int((res["own_score"] > res["best_neighbour_score"]).sum())
print("\norientation test -- cores whose tissue shape fits best in each orientation:")
for k in names:
    print(f"  {k:<14} {votes[k]:>4}  {'<- 02b chose this' if k == chosen else ''}")
print(f"label test: the own CellScape core fits better than all 8 neighbours for {own_wins} of {n} cores "
      f"({100 * own_wins / max(n, 1):.0f}%); median score own {res['own_score'].median():.2f}, "
      f"best neighbour {res['best_neighbour_score'].median():.2f}")
frac = votes[chosen] / max(n, 1)
if frac >= 0.6 and own_wins / max(n, 1) >= 0.6:
    print(f"VERDICT: orientation and labels confirmed ({100 * frac:.0f}% of cores vote for {chosen}).")
else:
    print("VERDICT: NOT confirmed -- look at the figures; send them to me before using the grid.")

In [ ]:
plot_summary(a.slide_id, res, votes, chosen, names, tiles, cs, cs_origin, cs_um, xen, xen_origin, b, both,
             cs_cores, L, half, grid_dir)
plot_gallery(a.slide_id, tiles, chosen, a.n_gallery, grid_dir)
print(f"\nwritten to {grid_dir}: {a.slide_id}_match_summary.png, {a.slide_id}_match_gallery.png, "
      f"{a.slide_id}_match_per_core.csv")